# Hunting MicroMonsters Bioinformatic Analysis Workflow

## Convert long *.fast5 names to short names

This document shows a method to easely convert long file names from ONT sequencing raw data to a short-consecutive-number set of files.
Each file name will be compound by the container folder name + the number of the file starting from 0.

#### Step-by-step breakdown:

1. find . -type f: Finds all files (not directories) starting from the current directory (.) and going into all subdirectories recursively.

2. while read -r filepath: Processes each file path found by the find command one at a time.

3. dir=$(dirname " $filepath"): Extracts the directory path from the full file path. For example:
    - If filepath is ./subfolder/myfile.txt, dir becomes ./subfolder
    - If filepath is ./myfile.txt, dir becomes .

4. file=$(basename "$filepath"): Extracts just the filename from the full path.
    - From ./subfolder/myfile.txt, file becomes myfile.txt
5. Folder name logic:
    - If the file is in the root directory (dir = "."), use the current working directory's name
    - Otherwise, use the subdirectory's name as the prefix
6. Counters calculation
    - Counts how many files already exist in that directory with the naming pattern foldername_*
    - This ensures proper numbering even if the script is run multiple times

7. Extension handling: Same as your original code - extracts the file extension.

8. File renaming: Moves the file to its new name with the pattern foldername_counter.extension


### Changing the name of each file in one folder

Change the name while keeping the original file extensions. This process usees bash kernel.

In [23]:
cd ~/Documents/HuntingMicromonsters/Testing_files/two_folders/1
counter=0
folder_name=$(basename "$PWD")
for file in *; do
    if [ -f "$file" ]; then
        # Get file extension
        ext="${file##*.}"
        # Only add extension if file had one
        if [ "$ext" != "$file" ]; then
            mv "$file" "${folder_name}_${counter}.${ext}"
        else
            mv "$file" "${folder_name}_${counter}"
        fi
        ((counter++))
    fi
done

### Changing the name of each file in all subdirectories of a folder with the format "folder_name + number + extension"

A modification from the previous script that can change the name of several subdirectories contained in a main folder.

In [33]:
cd ~/Documents/HuntingMicromonsters/fast5/pass

find . -type f | while read -r filepath; do
    # Get the directory and filename
    dir=$(dirname "$filepath")
    file=$(basename "$filepath")
    
    # Get folder name (use parent folder name or current folder for root)
    if [ "$dir" = "." ]; then
        folder_name=$(basename "$PWD")
    else
        folder_name=$(basename "$dir")
    fi
    
    # Count existing renamed files in this directory to continue numbering
    counter=$(find "$dir" -maxdepth 1 -name "${folder_name}_*" | wc -l)
    
    # Get file extension
    ext="${file##*.}"
    
    # Rename the file
    if [ "$ext" != "$file" ]; then
        mv "$filepath" "${dir}/${folder_name}_${counter}.${ext}"
    else
        mv "$filepath" "${dir}/${folder_name}_${counter}"
    fi
done

## Converting the *.fast5 files to pod5

In this section it is described the process to convert the fast5 files from the sequencing to pod5 to optimized computing performance. The data will be processed downstream using Guppy.